In [1]:
import numpy as np
import torch
from pathlib import Path

In [2]:
class biRNN(torch.nn.Module):
    def __init__(self, input_size: int, hidden_size: int, output_size: int,  layers: int = 1):
        super().__init__()

        self.layers = layers
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.Wxh_forward = torch.nn.ParameterList()
        self.Whh_forward = torch.nn.ParameterList()
        self.bh_forward = torch.nn.ParameterList()

        self.Wxh_backward = torch.nn.ParameterList()
        self.Whh_backward = torch.nn.ParameterList()
        self.bh_backward = torch.nn.ParameterList()

        for i in range(layers):
            current_input_size = input_size if i == 0 else hidden_size * 2

            self.Wxh_forward.append(torch.nn.Parameter(torch.randn(current_input_size, hidden_size) * 0.01))
            self.Whh_forward.append(torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01))
            self.bh_forward.append(torch.nn.Parameter(torch.zeros(hidden_size)))

            self.Wxh_backward.append(torch.nn.Parameter(torch.randn(current_input_size, hidden_size) * 0.01))
            self.Whh_backward.append(torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01))
            self.bh_backward.append(torch.nn.Parameter(torch.zeros(hidden_size)))

        self.why = torch.nn.Parameter(torch.randn(hidden_size * 2, output_size) * 0.01)
        self.by = torch.nn.Parameter(torch.zeros(output_size))

    def forward(self, x):
        batch_size = x.size(0)
        seq_len = x.size(1)

        for layer in range(self.layers):
            h_forward = torch.zeros(batch_size, self.hidden_size, device = x.device)
            h_backward = torch.zeros(batch_size, self.hidden_size, device = x.device)

            forward_output = []
            backward_output = []

            for t in range(seq_len):
                xt = x[:, t, :]

                h_forward = torch.tanh(xt @ self.Wxh_forward[layer] + h_forward @ self.Whh_forward[layer] + self.bh_forward[layer])

                forward_output.append(h_forward)

            for t in reversed(range(seq_len)):
                xt = x[:, t, :]

                h_backward = torch.tanh(xt @ self.Wxh_backward[layer] + h_backward @ self.Whh_backward[layer] + self.bh_backward[layer])

                backward_output.append(h_backward)

            backward_output.reverse()

            f_out_tensor = torch.stack(forward_output, dim = 1)
            b_out_tensor = torch.stack(backward_output, dim = 1)

            x = torch.cat([f_out_tensor, b_out_tensor], dim = -1)

        y = x @ self.why + self.by

        return y